# NativE Ablation on Merged-MMKG
This notebook reproduces the NativE baseline (Multi-modal Knowledge Graph Completion) on the newly unified `Merged-MMKG` dataset.

**INSTRUCTIONS FOR USER:**
1. Zip the `baselines` folder and the `data/Merged-MMKG` folder on your computer (You can delete the `images` folder inside `Merged-MMKG` before zipping to save upload time).
2. Upload this Zip file as a Kaggle Dataset.
3. Set the `KAGGLE_INPUT_DIR` below to point to the root of your uploaded files.

In [ ]:
import os

# TODO: Change this path to match your exact Kaggle dataset mount path
KAGGLE_INPUT_DIR = '/kaggle/input/merge-mmkg/proposed_mmkg'

# Copy NativE source code
!cp -r {KAGGLE_INPUT_DIR}/baselines/NativE /kaggle/working/NativE

# Create Data and Embeddings directories inside NativE
!mkdir -p /kaggle/working/NativE/data/Merged-MMKG
!mkdir -p /kaggle/working/NativE/embeddings

# Copy Dataset files (.txt files)
!cp {KAGGLE_INPUT_DIR}/data/Merged-MMKG/*.txt /kaggle/working/NativE/data/Merged-MMKG/

# Copy Pre-extracted Features (.pt files -> .pth files as expected by NativE)
!cp {KAGGLE_INPUT_DIR}/data/Merged-MMKG/ent_rgb_emb.pt /kaggle/working/NativE/embeddings/Merged-MMKG-visual.pth
!cp {KAGGLE_INPUT_DIR}/data/Merged-MMKG/ent_text_emb.pt /kaggle/working/NativE/embeddings/Merged-MMKG-text.pth
!cp {KAGGLE_INPUT_DIR}/data/Merged-MMKG/ent_depth_emb.pt /kaggle/working/NativE/embeddings/Merged-MMKG-numeric.pth

print("Data preparation complete!")


In [ ]:
%%bash
cd /kaggle/working/NativE/mmkgc
echo 'Recompiling C++ Data Loader for the current Kaggle CPU architecture...'
sed -i 's/\r$//' make.sh
sed -i 's/mkdir release/mkdir -p release/g' make.sh
bash make.sh


In [ ]:
%%python
import os
import re
# PATCHING THE HARDCODED EMBEDDING SIZES IN NativE'S 3-MODAL ARCHITECTURE
file_path = "/kaggle/working/NativE/mmkgc/module/model/AdvRelRotatEDB15K.py"
with open(file_path, "r") as f:
    content = f.read()

content = re.sub(
    r"self\.img_dim = 256.*?self\.img_embeddings = nn\.Embedding\.from_pretrained",
    "self.img_dim = img_emb.shape[1]\n        self.text_dim = text_emb.shape[1]\n        self.numeric_dim = numeric_emb.shape[1]\n\n        self.img_embeddings = nn.Embedding.from_pretrained",
    content,
    flags=re.DOTALL
)
content = content.replace("self.numeric_proj = nn.Sequential(\n            nn.Linear(self.text_dim, self.dim_e)", 
                          "self.numeric_proj = nn.Sequential(\n            nn.Linear(self.numeric_dim, self.dim_e)")
with open(file_path, "w") as f:
    f.write(content)
print("Patched NativE 3-Modal architecture to support Merged-MMKG embedding sizes!")


In [ ]:
%%python
import torch
import os
# CREATE EARLY FUSION (MEAN VISUAL + DEPTH) EMBEDDING
rgb = torch.load('/kaggle/working/NativE/embeddings/Merged-MMKG-visual.pth', map_location='cpu')
depth = torch.load('/kaggle/working/NativE/embeddings/Merged-MMKG-numeric.pth', map_location='cpu')
mean_emb = (rgb + depth) / 2.0
torch.save(mean_emb, '/kaggle/working/NativE/embeddings/Merged-MMKG-mean-visual.pth')
print("Created Early Fusion (Mean) embeddings for Ablation 1!")


In [ ]:
!cd /kaggle/working/NativE && sed -i 's/"link")/"link", type_constrain=False)/g' run_adv_wgan_gp.py
!cd /kaggle/working/NativE && sed -i 's/"link")/"link", type_constrain=False)/g' run_adv_wgan_gp_3modal.py
!echo 'Running CONTROL NativE (2 Modalities: Visual=RGB, Textual=Text. NO DEPTH)'
!cd /kaggle/working/NativE && python run_adv_wgan_gp.py -dataset=Merged-MMKG -batch_size=1024 -margin=12 -epoch=150 -dim=250 -adv_num=1 -save=Merged-MMKG-Control -neg_num=128 -mu=0.0001 -learning_rate=1e-4 -lrg=1e-4


In [ ]:
!echo 'Running ABLATION 1 (Early Fusion: Visual=Mean(RGB+Depth))'
!cd /kaggle/working/NativE && cp embeddings/Merged-MMKG-mean-visual.pth embeddings/Merged-MMKG-visual.pth
!cd /kaggle/working/NativE && python run_adv_wgan_gp.py -dataset=Merged-MMKG -batch_size=1024 -margin=12 -epoch=150 -dim=250 -adv_num=1 -save=Merged-MMKG-Ablation-Mean -neg_num=128 -mu=0.0001 -learning_rate=1e-4 -lrg=1e-4


In [ ]:
!echo 'Running ABLATION 2 (Late Fusion 3-Modal: Visual=RGB, Numeric=Depth, Textual=Text)'
!cd /kaggle/working/NativE && cp /kaggle/input/merge-mmkg/proposed_mmkg/data/Merged-MMKG/ent_rgb_emb.pt embeddings/Merged-MMKG-visual.pth
!cd /kaggle/working/NativE && python run_adv_wgan_gp_3modal.py -dataset=Merged-MMKG -batch_size=1024 -margin=12 -epoch=150 -dim=250 -adv_num=1 -save=Merged-MMKG-Ablation-3Modal -neg_num=128 -mu=0.0001 -learning_rate=1e-4 -lrg=1e-4
